# Bangladesh Constitution QA — BanglaBERT Fine-tuning
**Run this notebook on Google Colab Pro (A100 GPU recommended)**

Steps:
1. Install dependencies
2. Mount Google Drive
3. Build sample corpus (or upload your own)
4. Create SQuAD-format training data
5. Fine-tune BanglaBERT
6. Evaluate on test set
7. Run interactive demo

In [3]:
# ── Step 0: Check GPU ────────────────────────────────────────────────────────
!nvidia-smi
import torch
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

CUDA available: False


'nvidia-smi' is not recognized as an internal or external command,
operable program or batch file.


In [ ]:
# ── Step 1: Install dependencies ─────────────────────────────────────────────
!pip install -q transformers==4.40.0 datasets evaluate accelerate
!pip install -q rank-bm25 faiss-cpu sentence-transformers
!pip install -q bnlp-toolkit loguru wandb
!pip install -q git+https://github.com/csebuetnlp/normalizer

print('All dependencies installed')

In [ ]:
# ── Step 1: Mount Google Drive ───────────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import os

# Set project path
PROJECT_DIR = '/content/drive/MyDrive/bd_constitution_qa'
os.makedirs(PROJECT_DIR, exist_ok=True)
os.chdir(PROJECT_DIR)
print(f'✅ Working directory: {os.getcwd()}')

# ── Step 2: Add your JSON dataset ────────────────────────────────────────────

# Option A: If your JSON file is already in Google Drive
DATASET_PATH = os.path.join(PROJECT_DIR, 'data/processed/corpus.json')  # Recommended path for this project

# Option B: If your JSON is in another location in Drive, update this path
# DATASET_PATH = '/content/drive/MyDrive/your_folder/your_dataset.json'

print(f"📍 Your dataset should be at: {DATASET_PATH}")

# Check if the file exists
if os.path.exists(DATASET_PATH):
    print("✅ Dataset found!")
    # Load and show basic info
    import json
    with open(DATASET_PATH, 'r', encoding='utf-8') as f:
        data = json.load(f)
    
    print(f"Total items: {len(data) if isinstance(data, list) else 'Not a list'}")
    print("Sample keys:", list(data[0].keys()) if isinstance(data, list) and len(data) > 0 else "N/A")
else:
    print("❌ Dataset not found. Please upload it first.")

In [ ]:
# ── Step 3: Clone / copy project files ───────────────────────────────────────
# Option A: Clone from GitHub (recommended)
# !git clone https://github.com/YOUR_USERNAME/bd-constitution-qa.git .

# Option B: Upload files directly to Drive and copy
# (Upload requirements.txt and src/ folder to PROJECT_DIR in Drive)

# Option C: Create files inline (minimal setup)
import os
for d in ['data/raw', 'data/processed', 'data/squad', 'data/index',
          'models', 'src/preprocessing', 'src/retrieval', 'src/model']:
    os.makedirs(d, exist_ok=True)
print('Directory structure created')

In [ ]:
# ── Step 4: Build sample corpus ──────────────────────────────────────────────
# If you have the actual Constitution PDF:
#   from src.preprocessing.corpus_builder import ConstitutionCorpusBuilder
#   builder = ConstitutionCorpusBuilder(lang='bn')
#   builder.build_from_pdf('data/raw/constitution_bn.pdf')
#   builder.save('data/processed/corpus.json')

# Use built-in sample corpus for development:
from src.preprocessing.corpus_builder import build_sample_corpus
build_sample_corpus('data/processed/corpus.json')

import json
with open('data/processed/corpus.json') as f:
    corpus = json.load(f)
print(f'Corpus loaded: {len(corpus)} articles')
print('Sample:', corpus[0]['article_number'], '—', corpus[0]['title_en'])

In [ ]:
# ── Step 5: Build SQuAD training data ────────────────────────────────────────
from src.preprocessing.squad_builder import SQuADBuilder, BOOTSTRAP_QA_PAIRS

builder = SQuADBuilder('data/processed/corpus.json')
squad_data = builder.build_from_pairs(BOOTSTRAP_QA_PAIRS)
paths = builder.split_and_save(squad_data, 'data/squad/')

# Validate
for split, path in paths.items():
    report = builder.validate(path)
    print(f'{split}: {report}')

In [ ]:
# ── Step 6 (Optional): W&B tracking ─────────────────────────────────────────
# import wandb
# wandb.login()  # Enter your W&B API key when prompted
USE_WANDB = False  # Set True after login

In [ ]:
# ── Step 7: Fine-tune BanglaBERT ─────────────────────────────────────────────
from src.model.trainer import BanglaQATrainer

trainer = BanglaQATrainer(
    model_name='csebuetnlp/banglabert',
    output_dir='models/banglabert-qa',
    num_epochs=4,
    batch_size=8,
    learning_rate=2e-5,
    use_wandb=USE_WANDB,
)

metrics = trainer.train(
    train_path='data/squad/train.json',
    dev_path='data/squad/dev.json',
)

print('Training complete!')
print(f"Dev EM: {metrics.get('eval_exact_match', 0):.2f}")
print(f"Dev F1: {metrics.get('eval_f1', 0):.2f}")

In [ ]:
# ── Step 8: Build retrieval index ────────────────────────────────────────────
from src.retrieval.retriever import HybridRetriever

retriever = HybridRetriever(
    corpus_path='data/processed/corpus.json',
    cache_dir='data/index',
)
retriever.build_index(force_rebuild=True)

# Quick test
results = retriever.retrieve('বাংলাদেশের রাষ্ট্রধর্ম কী?', top_k=3)
for r in results:
    print(f"Article {r['article_number']}: {r['title_en']} (score: {r['retrieval_score']:.3f})")

In [ ]:
# ── Step 9: Evaluate on test set ─────────────────────────────────────────────
test_results = trainer.evaluate_on_test('data/squad/test.json')
print(f"Test EM: {test_results['exact_match']:.2f}")
print(f"Test F1: {test_results['f1']:.2f}")

# Retrieval recall
test_qa = json.load(open('data/squad/test.json'))
test_pairs = [
    {'question': qa['question'], 'article_number': para['context'][:5]}
    for art in test_qa['data']
    for para in art['paragraphs']
    for qa in para['qas']
][:20]  # sample
# retriever.evaluate_recall(test_pairs, top_k=5)

In [ ]:
# ── Step 10: Interactive demo ─────────────────────────────────────────────────
from src.model.inference import ConstitutionQAPipeline

pipeline = ConstitutionQAPipeline.load(
    model_path='models/banglabert-qa',
    corpus_path='data/processed/corpus.json',
)

test_questions = [
    'বাংলাদেশের রাষ্ট্রধর্ম কী?',
    'সংসদের মোট আসন সংখ্যা কত?',
    'রাষ্ট্রপতির বয়সসীমা কত?',
    'What is Article 7?',
    'সংবিধান সংশোধনে কত ভাগ ভোটের দরকার?',
]

print('='*60)
for q in test_questions:
    result = pipeline.answer(q)
    print(f'Q: {q}')
    print(f'A: {result.answer}')
    print(f'Source: Article {result.source_article_number} ({result.source_article_title_en})')
    print(f'Confidence: {result.confidence:.2%}')
    print('-'*60)

In [ ]:
# ── Step 11: Push model to HuggingFace Hub (optional) ───────────────────────
# from huggingface_hub import notebook_login
# notebook_login()
#
# from transformers import AutoModelForQuestionAnswering, AutoTokenizer
# model = AutoModelForQuestionAnswering.from_pretrained('models/banglabert-qa')
# tokenizer = AutoTokenizer.from_pretrained('models/banglabert-qa')
# model.push_to_hub('YOUR_USERNAME/banglabert-bd-constitution-qa')
# tokenizer.push_to_hub('YOUR_USERNAME/banglabert-bd-constitution-qa')